In [1]:
import os
import json
import cv2
import shutil
import random
import copy
from pathlib import Path
from collections import defaultdict
import albumentations as A
from tqdm import tqdm

# =====================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# =====================================================================
IN_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/train")
IN_IMG_DIR = IN_DIR / "images"
IN_META_PATH = IN_DIR / "metadata.jsonl"

OUT_ROOT = Path("/kaggle/working/BetterGoldDatasetV1")
OUT_TRAIN = OUT_ROOT / "train"
OUT_IMG_DIR = OUT_TRAIN / "images"
OUT_META_PATH = OUT_TRAIN / "metadata.jsonl"

DEBUG_DIR = Path("/kaggle/working/debug_better_gold")
ZIP_OUT_PATH = "/kaggle/working/BetterGoldDatasetV1" # shutil.make_archive tự thêm đuôi .zip

# Cấu hình BBOX_FORMAT (RUKOPYS gốc sử dụng [x, y, w, h] tức là 'coco')
BBOX_FORMAT = "coco" 
MAX_REPEATS = 4

# Phân loại Class theo nhóm
HEAD_CLASSES = {'handwritten', 'formula'}
MID_CLASSES = {'printed', 'annotation', 'table'}
TAIL_CLASSES = {'image', 'graph'}

# =====================================================================
# 2. HÀM CHUYỂN ĐỔI & KIỂM TRA BBOX
# =====================================================================
def original_bbox_to_coco(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]]
    raise ValueError(f"Unsupported format: {fmt}")

def coco_bbox_to_original(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
    raise ValueError(f"Unsupported format: {fmt}")

def clamp_coco_bbox(bbox, img_w, img_h):
    x, y, w, h = bbox
    x = max(0.0, float(x))
    y = max(0.0, float(y))
    w = max(1.0, min(float(w), float(img_w) - x))
    h = max(1.0, min(float(h), float(img_h) - y))
    return [x, y, w, h]

def inspect_bbox_schema():
    print("🔍 Kiểm tra cấu trúc Metadata mẫu...")
    with open(IN_META_PATH, 'r', encoding='utf-8') as f:
        for _ in range(5):
            record = json.loads(f.readline())
            regions = record.get('regions', [])
            if regions and 'bbox' in regions[0]:
                print(f"Sample file: {record.get('file_name')} | BBox mẫu: {regions[0]['bbox']}")
    print(f"Đã chốt định dạng BBOX_FORMAT đang dùng là: {BBOX_FORMAT}\n")

# =====================================================================
# 3. LOGIC TÁI CÂN BẰNG (RARE-RATIO & DOMINANT PENALTY)
# =====================================================================
def compute_image_priority(regions):
    counts = defaultdict(int)
    for r in regions:
        counts[r.get('type', 'unknown')] += 1
        
    total_boxes = sum(counts.values())
    if total_boxes == 0:
        return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    head_boxes = sum(counts[c] for c in HEAD_CLASSES)
    mid_boxes = sum(counts[c] for c in MID_CLASSES)
    tail_boxes = sum(counts[c] for c in TAIL_CLASSES)
    
    rare_ratio = tail_boxes / total_boxes
    dominant_ratio = head_boxes / total_boxes

    # Luật phạt bắt buộc (Dominant Penalty)
    if dominant_ratio >= 0.8 and tail_boxes <= 1:
        return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    # Tính điểm
    score = (
        tail_boxes * 3.0 + 
        mid_boxes * 1.2 + 
        rare_ratio * 8.0 + 
        counts['image'] * 2.0 + 
        counts['graph'] * 3.0
    ) - (
        dominant_ratio * 6.0 + 
        max(0, counts['handwritten'] - 20) * 0.15 + 
        max(0, counts['formula'] - 10) * 0.20
    )

    # Quyết định số lần Augment và loại Pipeline
    repeats = 0
    pipeline = "aug_light"

    if tail_boxes > 0 and rare_ratio >= 0.1:
        repeats = random.choice([3, 4])
        pipeline = "aug_rare"
    elif tail_boxes > 0 or mid_boxes > 5:
        repeats = random.choice([1, 2])
        pipeline = "aug_rare" if tail_boxes > 0 else "aug_dense"
    elif counts['handwritten'] >= 15:
        repeats = 1
        pipeline = "aug_dense"
    elif score > 0:
        repeats = random.choice([0, 1])
        pipeline = "aug_light"

    return {"repeats": min(repeats, MAX_REPEATS), "pipeline": pipeline, "stats": counts}

# =====================================================================
# 4. KHỞI TẠO AUGMENTATION PIPELINES
# =====================================================================
bbox_params = A.BboxParams(format='coco', label_fields=['region_idx'], min_visibility=0.5)

pipelines = {
    "aug_light": A.Compose([
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
        A.Blur(blur_limit=3, p=0.2)
    ], bbox_params=bbox_params),
    
    "aug_dense": A.Compose([
        A.Affine(rotate=(-3, 3), translate_percent={"x": (-0.02, 0.02), "y": (-0.02, 0.02)}, p=0.7),
        A.RandomBrightnessContrast(p=0.5),
        A.ISONoise(p=0.3)
    ], bbox_params=bbox_params),
    
    "aug_rare": A.Compose([
        A.Affine(rotate=(-3, 3), scale=(0.97, 1.03), p=0.8),
        A.RandomBrightnessContrast(p=0.6)
    ], bbox_params=bbox_params)
}

# =====================================================================
# 5. DỌN DẸP & XUẤT OUTPUT (CLEANUP CUỐI CÙNG)
# =====================================================================
def cleanup_working_except_zip(zip_path):
    print("🧹 Đang dọn dẹp toàn bộ dữ liệu trung gian trong /kaggle/working...")
    working_dir = Path("/kaggle/working")
    for item in working_dir.iterdir():
        if item.name == Path(zip_path).name:
            continue
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()
    print("✅ Cleanup hoàn tất. Chỉ giữ lại file Zip.")

# =====================================================================
# 6. CHƯƠNG TRÌNH CHÍNH
# =====================================================================
def main():
    if OUT_ROOT.exists(): shutil.rmtree(OUT_ROOT)
    if DEBUG_DIR.exists(): shutil.rmtree(DEBUG_DIR)
    
    OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
    DEBUG_DIR.mkdir(parents=True, exist_ok=True)
    
    inspect_bbox_schema()
    
    stats = {
        "original_images": 0, "augmented_images": 0, "failed_augments": 0, "invalid_boxes_dropped": 0,
        "boxes_before": defaultdict(int), "boxes_added": defaultdict(int)
    }
    
    with open(IN_META_PATH, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    out_meta_file = open(OUT_META_PATH, 'w', encoding='utf-8')
    debug_samples = []
    
    print("🚀 Bắt đầu Build Dataset BetterGoldDatasetV1...")
    for line in tqdm(lines):
        record = json.loads(line)
        fname_rel = record.get('file_name', '')
        fname = Path(fname_rel).name
        
        in_img_path = IN_IMG_DIR / fname
        if not in_img_path.exists(): continue
            
        img_w = record.get('image_width', 0)
        img_h = record.get('image_height', 0)
        regions = record.get('regions', [])
        
        # Thống kê ban đầu
        for r in regions: stats["boxes_before"][r.get('type', 'unknown')] += 1
            
        # -- BƯỚC 1: LƯU ẢNH GỐC --
        out_img_path = OUT_IMG_DIR / fname
        if not out_img_path.exists():
            os.symlink(in_img_path, out_img_path)
            
        out_meta_file.write(json.dumps(record, ensure_ascii=False) + '\n')
        stats["original_images"] += 1
        
        # Chuẩn hóa Box để Augment
        valid_bboxes = []
        valid_region_indices = []
        for idx, r in enumerate(regions):
            if 'bbox' in r and len(r['bbox']) == 4:
                coco_box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                clipped = clamp_coco_bbox(coco_box, img_w, img_h)
                if clipped[2] > 1 and clipped[3] > 1:
                    valid_bboxes.append(clipped)
                    valid_region_indices.append(idx)
                else:
                    stats["invalid_boxes_dropped"] += 1
        
        if not valid_bboxes: continue
            
        # -- BƯỚC 2: TÍNH ĐIỂM & QUYẾT ĐỊNH AUGMENT --
        priority = compute_image_priority(regions)
        repeats = priority["repeats"]
        pipeline_name = priority["pipeline"]
        transform = pipelines[pipeline_name]
        
        if repeats == 0: continue
            
        img = cv2.imread(str(in_img_path))
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # -- BƯỚC 3: THỰC THI AUGMENT --
        for i in range(repeats):
            try:
                augmented = transform(image=img, bboxes=valid_bboxes, region_idx=valid_region_indices)
            except Exception:
                stats["failed_augments"] += 1
                continue
                
            aug_bboxes = augmented['bboxes']
            aug_indices = augmented['region_idx']
            
            if not aug_bboxes: continue
                
            stem = Path(fname).stem
            ext = Path(fname).suffix
            new_fname = f"{stem}_{pipeline_name}_aug{i+1:03d}{ext}"
            
            cv2.imwrite(str(OUT_IMG_DIR / new_fname), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
            
            # Rebuild Record
            new_record = copy.deepcopy(record)
            new_record['file_name'] = f"images/{new_fname}"
            new_record['image_width'] = augmented['image'].shape[1]
            new_record['image_height'] = augmented['image'].shape[0]
            
            new_regions = []
            for new_bbox, orig_idx in zip(aug_bboxes, aug_indices):
                orig_idx = int(orig_idx) # Fix lỗi float index của Albumentations
                region = copy.deepcopy(regions[orig_idx])
                
                # Chuyển ngược từ Coco về Format gốc
                final_bbox = coco_bbox_to_original(new_bbox, BBOX_FORMAT)
                final_bbox = clamp_coco_bbox(final_bbox, new_record['image_width'], new_record['image_height'])
                
                region['bbox'] = [round(float(v), 2) for v in final_bbox]
                new_regions.append(region)
                stats["boxes_added"][region.get('type', 'unknown')] += 1
                
            new_record['regions'] = new_regions
            out_meta_file.write(json.dumps(new_record, ensure_ascii=False) + '\n')
            stats["augmented_images"] += 1
            
            # Chọn mẫu Debug
            if pipeline_name == "aug_rare" or (random.random() < 0.1 and len(debug_samples) < 20):
                if len(debug_samples) < 20:
                    debug_samples.append((OUT_IMG_DIR / new_fname, new_record))
                
    out_meta_file.close()
    
    # -- BƯỚC 4: VẼ DEBUG --
    print("\n🔍 Đang tạo Debug Visualization...")
    for img_p, rec in debug_samples:
        dbg_img = cv2.imread(str(img_p))
        if dbg_img is None: continue
        for r in rec.get('regions', []):
            box = r['bbox']
            # Chuyển về xyxy cho OpenCV vẽ
            if BBOX_FORMAT == "coco":
                x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[0]+box[2]), int(box[1]+box[3])
            else:
                x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
            cv2.rectangle(dbg_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(dbg_img, r.get('type',''), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.imwrite(str(DEBUG_DIR / img_p.name), dbg_img)

    # -- BƯỚC 5: IN BÁO CÁO THỐNG KÊ --
    print("\n📊 BÁO CÁO THỐNG KÊ CHẤT LƯỢNG:")
    print(f" - Ảnh gốc: {stats['original_images']}")
    print(f" - Ảnh augment tạo thêm: {stats['augmented_images']}")
    print(f" - Lỗi quá trình Augment: {stats['failed_augments']}")
    print(f" - Box bị loại do quá nhỏ (<1px) hoặc ngoài biên: {stats['invalid_boxes_dropped']}")
    
    print("\n📦 Phân bố Class (Trước -> Tăng thêm -> Final):")
    all_classes = set(stats['boxes_before'].keys()) | set(stats['boxes_added'].keys())
    for c in sorted(all_classes):
        b = stats['boxes_before'].get(c, 0)
        a = stats['boxes_added'].get(c, 0)
        print(f"   + {c.ljust(15)}: {str(b).rjust(6)} -> +{str(a).rjust(6)} -> = {b+a}")

    # -- BƯỚC 6: NÉN & DỌN DẸP --
    print(f"\n🗜️ Đang nén thành {ZIP_OUT_PATH}.zip...")
    shutil.make_archive(ZIP_OUT_PATH, 'zip', OUT_ROOT)
    
    cleanup_working_except_zip(f"{ZIP_OUT_PATH}.zip")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


🔍 Kiểm tra cấu trúc Metadata mẫu...
Sample file: images/09ba34df-2665-452c-9ef4-6998a5e7944c.jpg | BBox mẫu: [1213, 350, 2052, 530]
Sample file: images/3c54f7fd-3fe1-4f4b-8045-0efe0d656fbd.jpg | BBox mẫu: [603, 372, 2660, 508]
Sample file: images/2accc0fa-0368-4a59-9b16-ae107479c630.jpg | BBox mẫu: [755, 47, 1215, 145]
Sample file: images/2b9dbc5c-3dce-4c65-bcd2-261baea9dd7b.jpg | BBox mẫu: [1059, 484, 1764, 612]
Sample file: images/1fa2e2b7-74b2-4eda-b6dd-bd2cc046702c.jpg | BBox mẫu: [665, 182, 1516, 343]
Đã chốt định dạng BBOX_FORMAT đang dùng là: coco

🚀 Bắt đầu Build Dataset BetterGoldDatasetV1...


100%|██████████| 1330/1330 [00:44<00:00, 29.76it/s]



🔍 Đang tạo Debug Visualization...

📊 BÁO CÁO THỐNG KÊ CHẤT LƯỢNG:
 - Ảnh gốc: 1330
 - Ảnh augment tạo thêm: 157
 - Lỗi quá trình Augment: 0
 - Box bị loại do quá nhỏ (<1px) hoặc ngoài biên: 0

📦 Phân bố Class (Trước -> Tăng thêm -> Final):
   + annotation     :    484 -> +   186 -> = 670
   + formula        :   2950 -> +   345 -> = 3295
   + graph          :     24 -> +    51 -> = 75
   + handwritten    :  21523 -> +  1828 -> = 23351
   + image          :    117 -> +   204 -> = 321
   + printed        :    295 -> +   553 -> = 848
   + table          :    111 -> +    83 -> = 194

🗜️ Đang nén thành /kaggle/working/BetterGoldDatasetV1.zip...
🧹 Đang dọn dẹp toàn bộ dữ liệu trung gian trong /kaggle/working...
✅ Cleanup hoàn tất. Chỉ giữ lại file Zip.
